In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

import torch
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
PROJECT_ROOT = Path.cwd().parents[1]
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
EMB_OUT = DATA_PROCESSED / "drhp_embeddings"

EMB_OUT.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed data exists:", DATA_PROCESSED.exists())

Project root: /Users/jeeveshmishra/Desktop/IPO-Success-Prediction/IPO-Success-Prediction
Processed data exists: True


In [2]:
df = pd.read_csv(DATA_PROCESSED / "drhp_section_text.csv")

df.head()

,company_name,year,section,text,pdf_path,text_len
0,Rajputana Stainless Limited,2010,business,OUR BUSINESS 34\nSUMMARY OF OUR FINANCIAL INFO...,/Users/jeeveshmishra/Desktop/IPO-Success-Predi...,20000
1,Rajputana Stainless Limited,2010,risk,risk factors carefully before taking an invest...,/Users/jeeveshmishra/Desktop/IPO-Success-Predi...,20000
2,Rajputana Stainless Limited,2010,proceeds,OBJECTS OF THE ISSUE 62\nBASIS OF ISSUE PRICE ...,/Users/jeeveshmishra/Desktop/IPO-Success-Predi...,20000
3,The Promoters Of Our Company Are Bajaj Consume...,2010,business,OUR BUSINESS.....................................,/Users/jeeveshmishra/Desktop/IPO-Success-Predi...,20000
4,The Promoters Of Our Company Are Bajaj Consume...,2010,risk,risk factors carefully before taking an invest...,/Users/jeeveshmishra/Desktop/IPO-Success-Predi...,20000


In [3]:
df.shape

(842, 6)

In [4]:
df["section"].value_counts()

section
risk        311
business    302
proceeds    229
Name: count, dtype: int64

In [5]:
df["text_len"] = df["text"].str.len()
df["text_len"].describe()

count      842.000000
mean     19856.809976
std       1569.687287
min        751.000000
25%      20000.000000
50%      20000.000000
75%      20000.000000
max      20000.000000
Name: text_len, dtype: float64

In [6]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cpu'

In [7]:
FINBERT_MODEL = "ProsusAI/finbert"

finbert_tokenizer = AutoTokenizer.from_pretrained(FINBERT_MODEL)
finbert_model = AutoModel.from_pretrained(FINBERT_MODEL).to(DEVICE)
finbert_model.eval()

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

In [8]:
SBERT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

sbert_model = SentenceTransformer(SBERT_MODEL, device=DEVICE)

In [9]:
def finbert_embed(texts, batch_size=4, max_length=512):
    embeddings = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]

        encoded = finbert_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(DEVICE)

        with torch.no_grad():
            outputs = finbert_model(**encoded)
            cls_embeddings = outputs.last_hidden_state[:, 0, :]  # [CLS]

        embeddings.append(cls_embeddings.cpu().numpy())

    return np.vstack(embeddings)

In [10]:
finbert_embeddings = finbert_embed(df["text"].tolist())

finbert_embeddings.shape

  0%|          | 0/211 [00:00<?, ?it/s]

(842, 768)

In [11]:
sbert_embeddings = sbert_model.encode(
    df["text"].tolist(),
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True
)

sbert_embeddings.shape

Batches:   0%|          | 0/53 [00:00<?, ?it/s]

(842, 384)

In [12]:
emb_df = df[[
    "company_name",
    "year",
    "section"
]].copy()

emb_df["finbert_embedding"] = list(finbert_embeddings)
emb_df["sbert_embedding"] = list(sbert_embeddings)

emb_df.head()

,company_name,year,section,finbert_embedding,sbert_embedding
0,Rajputana Stainless Limited,2010,business,"[-0.19410531, 1.4739758, 0.06661505, -0.179569...","[0.0015715183, 0.029315984, -0.008042462, -0.0..."
1,Rajputana Stainless Limited,2010,risk,"[0.19888926, 1.2723527, -0.6584434, -0.5469876...","[-0.05275326, 0.09926484, -0.03792792, 0.02027..."
2,Rajputana Stainless Limited,2010,proceeds,"[-0.19195494, 1.582735, 0.021451645, -0.207139...","[-0.02480682, 0.021075375, -0.010292611, -0.00..."
3,The Promoters Of Our Company Are Bajaj Consume...,2010,business,"[0.41988212, 0.79313415, 0.6410486, 0.54914254...","[-0.023023352, 0.00429242, -0.07762703, -0.031..."
4,The Promoters Of Our Company Are Bajaj Consume...,2010,risk,"[0.17467354, 1.2288178, -0.68853724, -0.406259...","[-0.035153065, 0.006121801, -0.029135512, 0.02..."


In [13]:
SECTION_OUT = EMB_OUT / "drhp_section_embeddings.parquet"

emb_df.to_parquet(SECTION_OUT, index=False)

print("Saved:", SECTION_OUT)

Saved: /Users/jeeveshmishra/Desktop/IPO-Success-Prediction/IPO-Success-Prediction/data/processed/drhp_embeddings/drhp_section_embeddings.parquet
